# Document Question Answering System using Retrieval-Augmented Generation (RAG)

This project implements a simple Retrieval-Augmented Generation (RAG) system that answers user questions based on documents.
 Instead of relying only on a language model, the system first retrieves the most relevant information from the document and then generates an answer using that retrieved context.
 This helps produce more accurate and context-aware responses.

## Problem Statement

Develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents.
Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.

In [1]:
!pip -q install datasets sentence-transformers faiss-cpu transformers accelerate

In [2]:
!pip -q install datasets sentence-transformers faiss-cpu transformers accelerate gradio

In [3]:
!pip -q install pymupdf

In [4]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

In [5]:
import fitz
pdf_path = "/content/2102.04883v2.pdf"
document = fitz.open(pdf_path)
text = ""
for page in document:
    text += page.get_text()
document.close()
print("Text extracted successfully!")
print("Total characters:", len(text))

MuPDF error: syntax error: could not parse color space (826 0 R)

MuPDF error: syntax error: could not parse color space (896 0 R)

MuPDF error: syntax error: could not parse color space (931 0 R)

MuPDF error: syntax error: could not parse color space (970 0 R)

MuPDF error: syntax error: could not parse color space (1086 0 R)

MuPDF error: syntax error: could not parse color space (1171 0 R)

MuPDF error: syntax error: could not parse color space (1212 0 R)

MuPDF error: syntax error: could not parse color space (1255 0 R)

MuPDF error: syntax error: could not parse color space (1292 0 R)

MuPDF error: syntax error: could not parse color space (1369 0 R)

MuPDF error: syntax error: could not parse color space (1401 0 R)

MuPDF error: syntax error: could not parse color space (1477 0 R)

MuPDF error: syntax error: could not parse color space (1508 0 R)

MuPDF error: syntax error: could not parse color space (1563 0 R)

MuPDF error: syntax error: could not parse color space (1624 0 R)


In [6]:
chunk_size = 500
chunks = []
for i in range(0, len(text), chunk_size):
    chunk = text[i:i + chunk_size]
    chunks.append(chunk)
print("Total chunks created:", len(chunks))

Total chunks created: 411


In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [8]:
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embeddings shape: (411, 384)


The text chunks were converted into dense vector embeddings using the all-MiniLM-L6-v2 Sentence Transformer model. These embeddings capture the semantic meaning of each chunk and will be stored in the FAISS vector database for efficient similarity search.

In [9]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)

FAISS index created successfully!
Total vectors stored: 411


The generated embeddings were stored in a FAISS vector index. FAISS enables fast similarity search by comparing the user's query embedding with the stored document embeddings to retrieve the most relevant information.

In [10]:
def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    return retrieved_chunks

In [11]:
query = "What is supervised learning?"

results = retrieve_chunks(query)

for i, chunk in enumerate(results):
    print(f"\nChunk {i+1}\n")
    print(chunk)
    print("-" * 80)


Chunk 1

ience any classiﬁcation problem such as that of phases of matter,
document clustering, image compression (color reduction), etc.. In general, it helps
to build intuition about the data at hand.
9https://www.stat.cmu.edu/ larry/all-of-statistics/=data/faithful.dat
University of Zurich
16
Machine Learning for the Sciences
3
SUPERVISED LEARNING I
3
Supervised Learning without Neural
Networks
Supervised learning is the term for a machine learning task, where we are given
a dataset consisting of inpu
--------------------------------------------------------------------------------

Chunk 2

mation from the data and generate new
data resembling the data in the given dataset (unsupervised). However, the concept
of learning as commonly understood certainly encompasses other forms of learning
that are not falling into these data-driven categories.
An example for a form of learning not obviously covered by supervised or unsu-
pervised learning is learning how to walk: in particular, a c

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Language model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded successfully!


In [16]:
def generate_answer(question):

    if not question.strip():
        return "Please enter a question."

    # Retrieve top 2 relevant chunks
    retrieved = retrieve_chunks(question, top_k=2)

    context = "\n\n".join(retrieved)

    prompt = f"""
Use only the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if answer.strip() == "":
        return context

    return answer


In [17]:
question = "What is supervised learning?"
print(generate_answer(question))

a machine learning task


In [18]:
import gradio as gr

custom_css = """
.gradio-container{
    max-width:1000px !important;
    margin:auto;
}

footer{
    visibility:hidden;
}

h1{
    text-align:center;
}
"""

with gr.Blocks(
    title="RAG Document Question Answering System",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="cyan"
    ),
    css=custom_css
) as demo:

    gr.Markdown(
        """
# 📚 Smart RAG Document Assistant

Ask questions from your uploaded Machine Learning PDF using Retrieval-Augmented Generation (RAG).
"""
    )

    question = gr.Textbox(
        label="Enter your Question",
        placeholder="Example: What is overfitting?",
        lines=2
    )

    answer = gr.Textbox(
        label="Generated Answer",
        lines=12
    )

    ask = gr.Button("🔍 Generate Answer", variant="primary")

    ask.click(
        fn=generate_answer,
        inputs=question,
        outputs=answer
    )

    gr.Examples(
        examples=[
            ["What is overfitting?"],
            ["What is an autoencoder?"],
            ["Explain dimensionality reduction."],
            ["What is t-SNE?"]
        ],
        inputs=question
    )

demo.launch()

/tmp/ipykernel_23025/2571282481.py:18: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://305a061168ed90113c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
